# Determinanty sukcesu komercyjnego utworów muzycznych na platformie Spotify (2020–2025)

---

## 1. Wstęp i hipotezy badawcze

### Podsumowanie
Celem analizy jest ocena, co wiąże się z liczbą odtworzeń (`stream_count`) utworów w zbiorze *Spotify Streaming Data and Audio Feature (2020-25)*. Rozdzielam **cechy wewnętrzne utworu** (akustyka, tempo, czas trwania) od **czynników kontekstowych** (moment premiery, gatunek, kraj) oraz **skali artysty i wytwórni** (`artist_track_count`, `label`).

### Zasady przyjęte w analizie
- **Mediana zamiast średniej.** `stream_count` ma silnie prawostronny rozkład: kilka hitów windują średnią (średnia ≈ 7× mediana), dlatego porównania grup opieram na medianie i udziałach w sumie streamów.
- **Istotność praktyczna zamiast p-value.** Przy 50 000 wierszy nawet znikome różnice bywają „istotne statystycznie”. Zamiast testu oceniam **wielkość** różnicy: przyjęty próg to odchylenie mediany grupy o więcej niż **10%** od mediany ogólnej.
- **Jakość modeli oceniam poza próbą uczącą** (walidacja krzyżowa), a nie R² z dopasowania do tych samych danych.

### Hipotezy

| # | Nazwa | Opis |
|---|---|---|
| **H1** | **Koncentracja rynku** | Streamy są silnie skoncentrowane: niewielki odsetek utworów i artystów odpowiada za większość odtworzeń (zasada Pareto). |
| **H2** | **Audio vs skala artysty** | Cechy audio (`danceability`, `energy`, `tempo`...) mają mniejszą moc predykcyjną dla liczby streamów niż skala artysty i wytwórnia (`artist_track_count` – *liczba utworów artysty w zbiorze*, `label`). |
| **H3** | **Długość utworu** | (a) Mediana długości utworów w latach 2020–2025 maleje. (b) Krótsze utwory mają więcej streamów. |
| **H4** | **Strategia premiery** | Utwory wydawane w piątek, w weekend lub w określonych kwartałach mają wyraźnie wyższe mediany streamów (>10% względem mediany ogólnej). |

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 2. Wczytanie i kontrola jakości danych

In [ ]:
file = Path.cwd() / "spotify_artist_streaming_2020_2025.csv"

df = pd.read_csv(file)  # wymagane: filepath_or_buffer | opcjonalne: sep, usecols, dtype, parse_dates, nrows
mem_przed = df.memory_usage(deep=True).sum()  # deep=True liczy realny rozmiar kolumn tekstowych

print(f"Tabela posiada {df.shape[0]} wierszy oraz {df.shape[1]} kolumn.")
display(df.head())

### 2.1 Braki danych i duplikaty
Najpierw czyszczę teksty (spacje, puste tokeny), dopiero potem szukam braków – inaczej `" "` czy `"N/A"` zostałyby uznane za wartości.

In [ ]:
kolumny_tekstowe = df.select_dtypes(include=["object", "string"]).columns
tokeny_brakow = ["", " ", "N/A", "NA", "NULL", "null", "-", "None"]

df[kolumny_tekstowe] = df[kolumny_tekstowe].apply(lambda s: s.str.strip())
df[kolumny_tekstowe] = df[kolumny_tekstowe].replace(tokeny_brakow, pd.NA)

braki = df.isna().sum().loc[lambda s: s.gt(0)]
if braki.empty:
    print("Brak braków danych.")
else:
    print("Braki według kolumn:")
    display(braki.sort_values(ascending=False))

In [ ]:
liczba_duplikatow = df.duplicated().sum()
print(f"Liczba w pełni zduplikowanych wierszy: {liczba_duplikatow}")

# klucz naturalny: każdy utwór powinien występować raz
duplikaty_id = df["track_id"].duplicated().sum()
assert duplikaty_id == 0, f"Lista zawiera {duplikaty_id} zduplikowanych track_id."

### 2.2 Typy danych i pamięć
- Kolumny o niskiej liczbie unikalnych wartości → `category` (dni tygodnia i kwartały jako **uporządkowane**, żeby tabele i wykresy same sortowały się chronologicznie).
- Liczby → najmniejszy możliwy typ (`downcast`).
- Pamięć porównuję dla **całej tabeli** przed i po, więc oszczędność dotyczy realnie wszystkich zmian.

In [ ]:
dni_tygodnia = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
kwartaly = ["Q1", "Q2", "Q3", "Q4"]

df["release_date"] = pd.to_datetime(df["release_date"])  # wymagane: arg | opcjonalne: format, errors

kolumny_kategorie = ["genre", "popularity_category", "loudness_category", "country", "label", "artist_name"]
kolumny_string = ["track_id", "track_name", "album_name"]

df[kolumny_kategorie] = df[kolumny_kategorie].astype("category")
df[kolumny_string] = df[kolumny_string].astype("string")
df["release_day_of_week"] = pd.Categorical(df["release_day_of_week"], categories=dni_tygodnia, ordered=True)
df["release_quarter"] = pd.Categorical(df["release_quarter"], categories=kwartaly, ordered=True)

for kol in df.select_dtypes(include="integer").columns:
    df[kol] = pd.to_numeric(df[kol], downcast="integer")
for kol in df.select_dtypes(include="float").columns:
    df[kol] = pd.to_numeric(df[kol], downcast="float")

mem_po = df.memory_usage(deep=True).sum()
print(
    f"Pamięć całej tabeli: {mem_przed / 1024**2:.1f} MB → {mem_po / 1024**2:.1f} MB "
    f"(oszczędność {(1 - mem_po / mem_przed) * 100:.1f}%)"
)

### 2.3 Spójność kolumn pochodnych
Kolumny wyliczone z innych powinny się z nimi zgadzać. Sprawdzam to **przed** usunięciem kolumn źródłowych (`duration_ms`, `is_explicit_bool`).

In [ ]:
kontrole = {
    "explicit != is_explicit_bool": (df["explicit"] != df["is_explicit_bool"]).sum(),
    "release_year != rok z release_date": (df["release_date"].dt.year != df["release_year"]).sum(),
    "release_month != miesiąc z release_date": (df["release_date"].dt.month != df["release_month"]).sum(),
    "release_day_of_week != dzień z release_date": (df["release_date"].dt.day_name() != df["release_day_of_week"].astype(str)).sum(),
    "release_quarter != kwartał z release_date": (("Q" + df["release_date"].dt.quarter.astype(str)) != df["release_quarter"].astype(str)).sum(),
    "is_weekend_release != sobota/niedziela": (df["is_weekend_release"] != df["release_date"].dt.dayofweek.ge(5)).sum(),
    "log_stream_count != ln(1 + stream_count)": (~np.isclose(df["log_stream_count"], np.log1p(df["stream_count"]), atol=1e-3)).sum(),
    "duration_minutes != duration_ms / 60000": (~np.isclose(df["duration_minutes"], df["duration_ms"] / 60000, atol=0.01)).sum(),
}
# atol = dopuszczalna różnica bezwzględna: liczby dziesiętne rzadko są równe co do bitu, więc == zawodzi

display(pd.Series(kontrole, name="liczba_niezgodności").to_frame())
assert kontrole["explicit != is_explicit_bool"] == 0

### 2.4 Usunięcie zbędnych kolumn
- `duration_ms` – duplikat `duration_minutes`; `is_explicit_bool` – duplikat `explicit`.
- `key`, `key_name`, `mode`, `mode_name` – nie wchodzą do żadnej hipotezy.
- `popularity` – wskaźnik zdefiniowany przez Spotify, prawdopodobnie pochodna samych streamów, więc jako cecha objaśniająca byłby **wyciekiem informacji** (target leakage). Ta sama uwaga dotyczy `popularity_category` – kolumnę zostawiam tylko do kontroli poniżej i **nie używam jej w modelach**.

In [ ]:
kolumny_do_usuniecia = ["duration_ms", "key", "key_name", "mode", "mode_name", "popularity", "is_explicit_bool"]
df = df.drop(columns=kolumny_do_usuniecia)  # wymagane: labels/columns | opcjonalne: errors="ignore" (pomija brakujące)

# kontrola: czy popularity_category odzwierciedla streamy (jeśli tak → potwierdzenie wycieku)
display(
    df.groupby("popularity_category", observed=True)["stream_count"]
    .agg(liczba="count", mediana="median", minimum="min", maksimum="max")
)
df.info()

## 3. Opis danych i rozkłady (EDA)
`CV_%` liczę względem modułu średniej (dla `loudness` średnia jest ujemna).

In [ ]:
podsumowanie = df.describe(include="number").T
podsumowanie["range"] = podsumowanie["max"] - podsumowanie["min"]
podsumowanie["IQR"] = podsumowanie["75%"] - podsumowanie["25%"]
podsumowanie["CV_%"] = podsumowanie["std"] / podsumowanie["mean"].abs() * 100
podsumowanie

In [ ]:
# rok i miesiąc to znaczniki czasu, a stream_count/log_stream_count mają osobne wykresy niżej.
# artist_track_count jest cechą artysty powtórzoną w każdym jego utworze, więc rozkład jest ważony liczbą utworów.
kolumny_rozklad = df.select_dtypes(include="number").columns.drop(
    ["release_year", "release_month", "stream_count", "log_stream_count"]
)

for kol in kolumny_rozklad:
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    srednia, mediana, minimum = df[kol].mean(), df[kol].median(), df[kol].min()

    fig.suptitle(f"Rozkład zmiennej: {kol}", fontsize=14)

    sns.histplot(data=df, x=kol, ax=ax[0])  # wymagane: data + x (lub tylko x) | opcjonalne: bins, kde, ax
    ax[0].set_title("Histogram")
    ax[0].axvline(srednia, linestyle="--", color="red", label=f"Średnia: {srednia:.2f}")
    ax[0].axvline(mediana, linestyle="-.", color="green", label=f"Mediana: {mediana:.2f}")
    ax[0].axvline(minimum, linestyle="-", linewidth=0.5, color="pink", label=f"Minimum: {minimum:.2f}")
    ax[0].legend()

    sns.boxplot(data=df, x=kol, ax=ax[1])
    ax[1].set_title("Boxplot")

    plt.tight_layout()
    plt.show()

`stream_count` pokazuję w **skali logarytmicznej z biniami równymi w skali log** (liniowe biny na osi log dają zniekształcony histogram).

In [ ]:
x = df["stream_count"]
x = x[x > 0]  # log wymaga wartości > 0

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

biny = np.logspace(np.log10(x.min()), np.log10(x.max()), 60)
sns.histplot(x, bins=biny, ax=ax[0])
ax[0].set_xscale("log")
ax[0].axvline(x.median(), color="green", ls="-.", label=f"Mediana: {x.median():,.0f}")
ax[0].axvline(x.mean(), color="red", ls="--", label=f"Średnia: {x.mean():,.0f}")
ax[0].set_title("stream_count (skala log)")
ax[0].legend()

sns.boxplot(x=x, ax=ax[1])
ax[1].set_xscale("log")
ax[1].set_title("Boxplot (skala log)")

plt.tight_layout()
plt.show()

In [ ]:
kolumny_bool = df.select_dtypes(include="bool").columns
podsumowanie_bool = pd.DataFrame({
    "true": df[kolumny_bool].sum(),
    "true_%": df[kolumny_bool].mean() * 100,
    "false_%": (1 - df[kolumny_bool].mean()) * 100,
})
display(podsumowanie_bool)

kolumny_kat = df.select_dtypes(include=["string", "category"]).columns
podsumowanie_kat = pd.DataFrame({
    "unikalne": df[kolumny_kat].nunique(),
    "unikalne_%": df[kolumny_kat].nunique() / len(df) * 100,
    "top_1": df[kolumny_kat].agg(lambda s: s.value_counts().index[0]),
    "top_1_%": df[kolumny_kat].agg(lambda s: s.value_counts(normalize=True).iloc[0] * 100),
})
podsumowanie_kat

## 4. Struktura katalogu: gatunki i artyści
Sama liczba utworów nie mówi o sukcesie komercyjnym, dlatego obok niej pokazuję **udział w streamach** i **medianę streamów na utwór**.

In [ ]:
agg_gatunek = (
    df.groupby("genre", observed=True)  # observed=True: pomija puste kategorie (bez tego FutureWarning)
    .agg(
        liczba_utworow=("track_id", "nunique"),
        suma_streamow=("stream_count", "sum"),
        mediana_streamow=("stream_count", "median"),
    )
    .assign(
        udzial_utworow_pct=lambda t: t["liczba_utworow"] / t["liczba_utworow"].sum() * 100,
        udzial_streamow_pct=lambda t: t["suma_streamow"] / t["suma_streamow"].sum() * 100,
    )
    .sort_values("liczba_utworow", ascending=False)
)
display(agg_gatunek.round(1))

kolejnosc_gatunkow = agg_gatunek.index.tolist()
fig, ax = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
sns.barplot(data=agg_gatunek.reset_index(), y="genre", x="udzial_utworow_pct", order=kolejnosc_gatunkow, ax=ax[0])
ax[0].set_title("Udział w liczbie utworów [%]")
sns.barplot(data=agg_gatunek.reset_index(), y="genre", x="mediana_streamow", order=kolejnosc_gatunkow, ax=ax[1])
ax[1].set_title("Mediana streamów na utwór")
ax[1].set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
agg_artysta = (
    df.groupby("artist_name", observed=True)
    .agg(
        liczba_utworow=("track_id", "nunique"),
        suma_streamow=("stream_count", "sum"),
        mediana_streamow=("stream_count", "median"),
    )
    .assign(
        udzial_utworow_pct=lambda t: t["liczba_utworow"] / t["liczba_utworow"].sum() * 100,
        udzial_streamow_pct=lambda t: t["suma_streamow"] / t["suma_streamow"].sum() * 100,
    )
    .sort_values("suma_streamow", ascending=False)
)
display(agg_artysta.head(10).round(1))

# czym jest artist_track_count? sprawdzam, czy to po prostu liczba wierszy artysty w tym zbiorze
zgodne = (df["artist_track_count"] == df.groupby("artist_name", observed=True)["track_id"].transform("size")).all()
print(f"artist_track_count == liczba utworów artysty w zbiorze: {bool(zgodne)}")

> `artist_track_count` (jeśli powyższa kontrola daje `True`) to **rozmiar katalogu artysty w tym zbiorze**, a nie jego popularność czy liczba słuchaczy. W H2 traktuję go więc jako przybliżenie „skali artysty”, a nie „marki”.

## 5. H1: koncentracja streamów
Sprawdzam koncentrację na dwóch poziomach: **utworów** i **artystów**. Ten drugi poziom jest istotny, bo artyści z dużym katalogiem mają więcej „losowań” na hit.

In [ ]:
# --- poziom utworów ---
s = np.sort(df["stream_count"].to_numpy(dtype="int64"))[::-1]  # od największych
razem = s.sum()
progi = [1, 5, 10, 20, 50]

udzialy = pd.Series(
    {f"top {p}%": s[: int(len(s) * p / 100)].sum() / razem * 100 for p in progi},
    name="udział w streamach [%]",
)
display(udzialy.round(1).to_frame())

# --- poziom artystów ---
s_art = agg_artysta["suma_streamow"].to_numpy(dtype="int64")  # tabela jest już posortowana malejąco
razem_art = s_art.sum()

udzial_artystow = pd.Series(
    {f"top {p}% artystów": s_art[: max(1, int(len(s_art) * p / 100))].sum() / razem_art * 100 for p in [1, 5, 10, 20]}
    | {"top 10 artystów": s_art[:10].sum() / razem_art * 100, "top 50 artystów": s_art[:50].sum() / razem_art * 100},
    name="udział w streamach [%]",
)
display(udzial_artystow.round(1).to_frame())

In [ ]:
# krzywa koncentracji: skumulowany udział w streamach, gdy utwory ustawimy od najpopularniejszych
os_x = np.arange(1, len(s) + 1) / len(s) * 100
os_y = np.cumsum(s) / razem * 100

plt.figure(figsize=(8, 6))
plt.plot(os_x, os_y, lw=2, label="Krzywa koncentracji")
plt.plot([0, 100], [0, 100], "r--", label="Równy podział")
for p in (1, 10, 20):
    plt.scatter(p, udzialy[f"top {p}%"], color="black", zorder=3)
    plt.annotate(f"top {p}% → {udzialy[f'top {p}%']:.0f}%", (p, udzialy[f"top {p}%"]),
                 textcoords="offset points", xytext=(10, -12))
plt.xlabel("Udział najpopularniejszych utworów [%]")
plt.ylabel("Skumulowany udział w streamach [%]")
plt.title("Koncentracja streamów na poziomie utworów")
plt.legend()
plt.tight_layout()
plt.show()

Czy udział artysty w streamach wynika z jakości utworów, czy z liczby utworów? Porównanie **udziału w utworach** z **udziałem w streamach** dla największych artystów (tabela wyżej, kolumny `udzial_utworow_pct` vs `udzial_streamow_pct`): jeśli oba są zbliżone, przewaga artysty to głównie skala katalogu; jeśli streamy są wyraźnie wyższe, utwory artysty są też mocniejsze.

## 6. H2: cechy audio vs skala artysty
### 6.1 Zależności jednowymiarowe
`upbeat_score` jest silnie skorelowany z `danceability`, `energy` i `loudness` (wygląda na kompozyt tych cech), więc **nie wchodzi do modeli** – zdublowałby informację i zaburzył interpretację współczynników.

In [ ]:
cechy_audio = ["danceability", "energy", "loudness", "instrumentalness", "tempo", "duration_minutes"]

korelacje = df[cechy_audio + ["upbeat_score", "artist_track_count", "log_stream_count"]].corr(method="spearman")
# method: "pearson" (domyślnie, zależności liniowe) | "spearman" (rangowa, odporna na skośność i outliery)

plt.figure(figsize=(9, 7))
sns.heatmap(korelacje, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5)
plt.title("Korelacja Spearmana: cechy audio, skala artysty i log_stream_count")
plt.tight_layout()
plt.show()

display(korelacje["log_stream_count"].drop("log_stream_count").sort_values(key=abs, ascending=False).round(3).to_frame("rho ze streamami"))

In [ ]:
# profil audio gatunków; każdą cechę standaryzuję (z-score między gatunkami), bo tempo (~120) i danceability (~0.7) mają różne skale
profil_gatunkow = df.groupby("genre", observed=True)[["danceability", "energy", "loudness", "tempo"]].mean()
profil_z = (profil_gatunkow - profil_gatunkow.mean()) / profil_gatunkow.std()

plt.figure(figsize=(9, 6))
sns.heatmap(profil_z, annot=True, fmt=".1f", cmap="RdBu_r", center=0)
plt.title("Profil audio gatunków (z-score między gatunkami)")
plt.ylabel("Gatunek")
plt.tight_layout()
plt.show()

In [ ]:
# skala artysty vs typowy wynik jego utworów (jeden punkt = jeden artysta)
rho_artysta = agg_artysta[["liczba_utworow", "mediana_streamow"]].corr(method="spearman").iloc[0, 1]

plt.figure(figsize=(8, 6))
sns.scatterplot(data=agg_artysta, x="liczba_utworow", y="mediana_streamow", alpha=0.6)
plt.xscale("log")
plt.yscale("log")
plt.title(f"Rozmiar katalogu artysty vs mediana streamów utworu (Spearman ρ = {rho_artysta:.2f})")
plt.xlabel("Liczba utworów artysty w zbiorze (log)")
plt.ylabel("Mediana streamów na utwór (log)")
plt.tight_layout()
plt.show()

### 6.2 Modele: ile wariancji tłumaczą kolejne grupy cech
Zmienna objaśniana: `log_stream_count`. Zestawy cech są **kumulatywne**:

| Zestaw | Cechy |
|---|---|
| M1 | audio (bez `upbeat_score`) |
| M2 | M1 + premiera (`release_year`, `release_quarter`, `is_weekend_release`) |
| M3 | M2 + `genre`, `country` |
| M4 | M3 + skala i marka (`log1p(artist_track_count)`, `label`) |

Co zmieniłem względem prostego OLS na całych danych:
- **R² z walidacji krzyżowej (5-fold)**, a nie z dopasowania in-sample. R² in-sample zawsze rośnie po dodaniu zmiennych (a M4 ma kilkadziesiąt dummies), więc porównywanie zestawów na nim jest stronnicze.
- **Kodowanie one-hot wewnątrz `Pipeline`**, więc w każdym foldzie uczy się je tylko na danych treningowych (bez przecieku).
- **Dwa schematy podziału:** losowy (`KFold`) i **grupowany po artyście** (`GroupKFold`). W losowym ten sam artysta trafia i do treningu, i do testu, więc `artist_track_count` działa jak identyfikator artysty. Grupowany mierzy, czy skala artysty pomaga przy **nowych** artystach.
- `artist_track_count` po `log1p` (skośny rozkład).

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold, KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

df_model = df.assign(
    log_artist_track_count=np.log1p(df["artist_track_count"]),
    is_weekend_release=df["is_weekend_release"].astype(int),
)

cechy_premiera = ["release_year", "release_quarter", "is_weekend_release"]
cechy_gatunek_kraj = ["genre", "country"]
cechy_marka = ["log_artist_track_count", "label"]
kolumny_kategoryczne = ["release_quarter", "genre", "country", "label"]

zestawy = {
    "M1 audio": cechy_audio,
    "M2 + premiera": cechy_audio + cechy_premiera,
    "M3 + gatunek i kraj": cechy_audio + cechy_premiera + cechy_gatunek_kraj,
    "M4 + marka i katalog": cechy_audio + cechy_premiera + cechy_gatunek_kraj + cechy_marka,
}

y = df_model["log_stream_count"]
grupy = df_model["artist_name"]

cv_losowy = KFold(n_splits=5, shuffle=True, random_state=42)  # wymagane: n_splits | opcjonalne: shuffle, random_state
cv_artysta = GroupKFold(n_splits=5)                            # wymagane: n_splits; grupy przekazuje się w cross_val_score(groups=...)

wyniki = {}
for nazwa, cechy in zestawy.items():
    kat = [c for c in cechy if c in kolumny_kategoryczne]
    przygotowanie = ColumnTransformer(
        [("kat", OneHotEncoder(handle_unknown="ignore"), kat)],  # handle_unknown="ignore": nowa kategoria w teście → same zera
        remainder="passthrough",                                   # pozostałe (numeryczne) kolumny bez zmian
    )
    model = Pipeline([("prep", przygotowanie), ("lr", LinearRegression())])
    X = df_model[cechy]

    r2_los = cross_val_score(model, X, y, cv=cv_losowy, scoring="r2")
    r2_art = cross_val_score(model, X, y, cv=cv_artysta, groups=grupy, scoring="r2")
    wyniki[nazwa] = {
        "r2_losowy": r2_los.mean(), "r2_losowy_std": r2_los.std(),
        "r2_artysta": r2_art.mean(), "r2_artysta_std": r2_art.std(),
    }

wyniki = pd.DataFrame(wyniki).T
display(wyniki.round(4))  # punkt odniesienia: model przewidujący zawsze średnią ma R² = 0

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
pozycje = np.arange(len(wyniki))
szerokosc = 0.38
ax.bar(pozycje - szerokosc / 2, wyniki["r2_losowy"], szerokosc, yerr=wyniki["r2_losowy_std"], capsize=3, label="podział losowy")
ax.bar(pozycje + szerokosc / 2, wyniki["r2_artysta"], szerokosc, yerr=wyniki["r2_artysta_std"], capsize=3, label="podział po artyście (nowi artyści)")
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(pozycje)
ax.set_xticklabels(wyniki.index, rotation=15)
ax.set_ylabel("R² (walidacja krzyżowa)")
ax.set_title("Moc predykcyjna kolejnych zestawów cech")
ax.legend()
plt.tight_layout()
plt.show()

### 6.3 Random Forest i ważność cech
Random Forest sprawdza, czy pominięte przez model liniowy **nieliniowości i interakcje** coś wnoszą. Dwie zmiany względem domyślnej konfiguracji:
- `min_samples_leaf=50` ogranicza dopasowanie do szumu (domyślne drzewa rosną do liści z 1 obserwacją). W realnym projekcie ten parametr się stroi, np. `RandomizedSearchCV`.
- Ważność cech liczę metodą **permutation importance na zbiorze testowym** i **na oryginalnych kolumnach** (nie na dummies). Wbudowane `feature_importances_` (impurity) faworyzuje zmienne ciągłe o wielu unikalnych wartościach, więc przy modelu bez mocy predykcyjnej pokazuje ranking szumu.

In [ ]:
X_all = df_model[zestawy["M4 + marka i katalog"]]
X_train, X_test, y_train, y_test = train_test_split(X_all, y, test_size=0.2, random_state=42)

kat = [c for c in X_all.columns if c in kolumny_kategoryczne]
rf = Pipeline([
    ("prep", ColumnTransformer([("kat", OneHotEncoder(handle_unknown="ignore"), kat)], remainder="passthrough")),
    ("rf", RandomForestRegressor(n_estimators=200, min_samples_leaf=50, n_jobs=-1, random_state=42)),
])
rf.fit(X_train, y_train)

r2_rf_train = rf.score(X_train, y_train)
r2_rf_test = rf.score(X_test, y_test)
print(f"Random Forest R² – trening: {r2_rf_train:.4f} | test: {r2_rf_test:.4f}")

# permutation importance: o ile spada R² na teście, gdy losowo przetasuję jedną kolumnę
# wymagane: estimator, X, y | opcjonalne: scoring, n_repeats (domyślnie 5), random_state, n_jobs
perm = permutation_importance(rf, X_test, y_test, scoring="r2", n_repeats=10, random_state=42, n_jobs=-1)
waznosc = pd.Series(perm.importances_mean, index=X_test.columns)
blad = pd.Series(perm.importances_std, index=X_test.columns)
kolejnosc = waznosc.sort_values(ascending=True).index

plt.figure(figsize=(9, 6))
plt.barh(kolejnosc, waznosc[kolejnosc], xerr=blad[kolejnosc], capsize=3)
plt.axvline(0, color="black", lw=0.8)
plt.xlabel("Spadek R² po przetasowaniu cechy (test)")
plt.title("Permutation importance – Random Forest (zestaw M4)")
plt.tight_layout()
plt.show()

## 7. H3: długość utworu
Trend pokazuję jako **medianę z pasmem międzykwartylowym (Q1–Q3) na osi zaczynającej się od zera**. Wykres z osią dopasowaną do danych potrafi zamienić różnicę kilku sekund w dramatyczny spadek.

In [ ]:
dlugosc_rok = (
    df.groupby("release_year")["duration_minutes"]
    .quantile([0.25, 0.5, 0.75])
    .unstack()
    .rename(columns={0.25: "q25", 0.5: "mediana", 0.75: "q75"})
)
display(dlugosc_rok.round(2))

zmiana_pierwszy_ostatni_s = (dlugosc_rok["mediana"].iloc[-1] - dlugosc_rok["mediana"].iloc[0]) * 60
rozrzut_median_s = (dlugosc_rok["mediana"].max() - dlugosc_rok["mediana"].min()) * 60
print(f"Zmiana mediany {dlugosc_rok.index[0]}→{dlugosc_rok.index[-1]}: {zmiana_pierwszy_ostatni_s:+.1f} s | rozstęp median między latami: {rozrzut_median_s:.1f} s")

plt.figure(figsize=(10, 5))
plt.fill_between(dlugosc_rok.index, dlugosc_rok["q25"], dlugosc_rok["q75"], alpha=0.25, color="teal", label="Q1–Q3")
plt.plot(dlugosc_rok.index, dlugosc_rok["mediana"], marker="o", lw=2.5, color="teal", label="Mediana")
plt.ylim(0, dlugosc_rok["q75"].max() * 1.2)
plt.xticks(dlugosc_rok.index)
plt.title("Długość utworów wg roku wydania (2020–2025)")
plt.xlabel("Rok wydania")
plt.ylabel("Czas trwania (minuty)")
plt.legend()
plt.show()

In [ ]:
# część (b): mediana streamów w decylach długości utworu (10 równolicznych przedziałów)
decyl_dlugosci = pd.qcut(df["duration_minutes"], q=10, duplicates="drop")  # wymagane: x, q | opcjonalne: labels, duplicates
mediana_wg_decyla = df.groupby(decyl_dlugosci, observed=True)["stream_count"].median()
mediana_ogolna = df["stream_count"].median()

rho_dlugosc = df[["duration_minutes", "log_stream_count"]].corr(method="spearman").iloc[0, 1]

plt.figure(figsize=(11, 5))
plt.bar(range(len(mediana_wg_decyla)), mediana_wg_decyla.to_numpy())
plt.xticks(range(len(mediana_wg_decyla)), [f"{i.left:.1f}–{i.right:.1f}" for i in mediana_wg_decyla.index], rotation=30)
plt.axhline(mediana_ogolna, color="red", ls="--", label=f"Mediana ogólna: {mediana_ogolna:,.0f}")
plt.title(f"Mediana streamów wg decyli długości utworu (Spearman ρ = {rho_dlugosc:.3f})")
plt.xlabel("Długość utworu [min]")
plt.ylabel("Mediana streamów")
plt.legend()
plt.tight_layout()
plt.show()

## 8. H4: strategia premiery
Dla każdego wymiaru (dzień tygodnia, kwartał, weekend) liczę mediany i **procentowe odchylenie mediany grupy od mediany ogólnej**. Wykresy mają wspólną, symetryczną skalę ±20% (lub większą, jeśli dane tego wymagają), aby widać było, czy różnice przekraczają przyjęty próg 10%.
Średnią podaję pomocniczo: pojedyncze hity potrafią ją zmienić o kilkanaście procent bez wpływu na typowy utwór.

In [ ]:
grupy_premiery = {
    "Dzień tygodnia": "release_day_of_week",
    "Kwartał": "release_quarter",
    "Weekend": "is_weekend_release",
}

tabele_premiery = {}
for etykieta, kol in grupy_premiery.items():
    tabela = (
        df.groupby(kol, observed=True)["stream_count"]
        .agg(liczba="count", mediana="median", srednia="mean", suma="sum")
        .assign(
            udzial_streamow_pct=lambda t: t["suma"] / t["suma"].sum() * 100,
            odch_mediany_pct=lambda t: (t["mediana"] / mediana_ogolna - 1) * 100,
        )
    )
    tabele_premiery[etykieta] = tabela
    print(f"--- {etykieta} ---")
    display(tabela.drop(columns="suma").round(1))

max_odch = max(t["odch_mediany_pct"].abs().max() for t in tabele_premiery.values())
lim = max(20, np.ceil(max_odch * 1.2))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, (etykieta, tabela) in zip(axes, tabele_premiery.items()):
    ax.bar(tabela.index.astype(str), tabela["odch_mediany_pct"])
    ax.axhline(0, color="black", lw=0.8)
    ax.axhspan(-10, 10, color="green", alpha=0.08, label="próg ±10%")
    ax.set_ylim(-lim, lim)
    ax.set_title(etykieta)
    ax.tick_params(axis="x", rotation=30)
axes[0].set_ylabel("Odchylenie mediany od mediany ogólnej [%]")
axes[0].legend()
plt.tight_layout()
plt.show()

piatek = df["release_day_of_week"].eq("Friday")
med_piatek = df.loc[piatek, "stream_count"].median()
med_reszta = df.loc[~piatek, "stream_count"].median()
print(f"Mediana: piątek {med_piatek:,.0f} vs pozostałe dni {med_reszta:,.0f} ({(med_piatek / med_reszta - 1) * 100:+.1f}%)")

## 9. Podsumowanie i weryfikacja hipotez
Wnioski poniżej są **wyliczane z wyników powyżej** (nie wpisane ręcznie), więc zmienią się razem z danymi lub parametrami analizy.

In [ ]:
# --- H1 ---
h1_top10, h1_top20 = udzialy["top 10%"], udzialy["top 20%"]
h1_art = udzial_artystow["top 10% artystów"]
h1 = "POTWIERDZONA" if h1_top20 >= 80 else "CZĘŚCIOWO" if h1_top10 >= 50 else "NIEPOTWIERDZONA"

# --- H2 ---
r2_audio = wyniki.loc["M1 audio", "r2_losowy"]
r2_pelny = wyniki.loc["M4 + marka i katalog", "r2_losowy"]
r2_pelny_art = wyniki.loc["M4 + marka i katalog", "r2_artysta"]
przyrost = r2_pelny - r2_audio
if r2_pelny < 0.01:
    h2 = "NIEPOTWIERDZONA: żaden zestaw cech (audio ani skala artysty) nie przewiduje streamów"
elif przyrost > 0.01:
    h2 = "KIERUNKOWO ZGODNA: skala artysty dodaje więcej niż audio, ale wyjaśnia niewiele"
else:
    h2 = "NIEPOTWIERDZONA: skala artysty nie dodaje istotnie ponad audio"

# --- H3 ---
trend_maleje = (zmiana_pierwszy_ostatni_s < 0) and (rozrzut_median_s >= 10)
zwiazek_ujemny = rho_dlugosc < -0.05
h3 = "POTWIERDZONA" if (trend_maleje and zwiazek_ujemny) else "CZĘŚCIOWO" if (trend_maleje or zwiazek_ujemny) else "NIEPOTWIERDZONA"

# --- H4 ---
h4 = "NIEPOTWIERDZONA (brak praktycznie istotnych różnic)" if max_odch < 10 else "CZĘŚCIOWO/POTWIERDZONA (są grupy powyżej progu, patrz tabele)"

print("=== WERYFIKACJA HIPOTEZ ===")
print(f"H1 [{h1}] Top 10% utworów = {h1_top10:.1f}% streamów, top 20% = {h1_top20:.1f}%; top 10% artystów = {h1_art:.1f}%.")
print(f"H2 [{h2}]")
print(f"    R² (CV, podział losowy): audio = {r2_audio:.4f}, pełny model = {r2_pelny:.4f} (przyrost {przyrost * 100:+.2f} p.p.).")
print(f"    R² (CV, nowi artyści): pełny model = {r2_pelny_art:.4f}. Random Forest, test: {r2_rf_test:.4f}.")
print(f"H3 [{h3}] Zmiana mediany długości {zmiana_pierwszy_ostatni_s:+.1f} s, rozstęp median między latami {rozrzut_median_s:.1f} s; Spearman długość vs streamy = {rho_dlugosc:.3f}.")
print(f"H4 [{h4}] Największe odchylenie mediany grupy od ogólnej: {max_odch:.1f}% (próg 10%).")

### Ograniczenia
- **Dane wyglądają na syntetyczne** (500 artystów, 30 wytwórni i krajów, równomierne dni tygodnia, praktycznie zerowe korelacje cech audio). Warto zweryfikować pochodzenie zbioru; wnioski dotyczą **tego zbioru**, a nie platformy Spotify.
- **Zależności obserwacyjne:** nawet przy istotnym związku nie wynika z niego przyczynowość.
- **Niski poziom wyjaśnionej wariancji** oznacza, że zebrane cechy nie opisują większości zróżnicowania streamów. Brakuje np. działań promocyjnych, playlist, historii artysty.
- `artist_track_count` opisuje rozmiar katalogu w zbiorze, a nie popularność artysty.
- Próg istotności praktycznej (10%) jest założeniem analitycznym; przy innym progu zmieni się werdykt H4.

### Możliwe kolejne kroki
1. **Model na poziomie artysty** (jeden wiersz = artysta): czy skala katalogu i wytwórnia tłumaczą medianę streamów lepiej niż na poziomie utworów?
2. **Analiza wytwórni** (`label`): udział w streamach vs udział w liczbie utworów, analogicznie do artystów.
3. **Porównanie z modelem gradient boosting** (`HistGradientBoostingRegressor`), obsługuje kategorie natywnie i szybko liczy się na 50 tys. wierszy; przy braku sygnału potwierdzi wnioski z Random Forest.